In [6]:
from dotenv import load_dotenv
load_dotenv()

True

In [ ]:
from langchain.chat_models import init_chat_model

llm = init_chat_model("google_genai:gemini-2.5-flash-lite", temperature=0)
response = llm.invoke("Hola, como estas?")
response.pretty_print()

================================== Ai Message ==================================

¡Hola! Estoy muy bien, gracias por preguntar. ¿Y tú, cómo estás?


In [ ]:
system_prompt = """
Eres un asistente de ventas que ayuda a los clientes a encontrar productos que necesitan.

Y tus posibles productos son:
- Teléfonos inteligentes
- Computadoras portátiles
- Auriculares
- Relojes inteligentes
"""
messages = [
    ("system", system_prompt),
    ("user", "Dime los productos que ofreces."),
]

response = llm.invoke(messages)
response.pretty_print()

In [ ]:
from langchain_core.tools import tool

@tool
def get_products():
    "Obtiene la lista de productos disponibles."
    return [
        "Teléfonos inteligentes",
        "Computadoras portátiles",
        "Auriculares",
        "Relojes inteligentes"
    ]

In [24]:
from langchain_core.tools import tool

@tool("get_products", description="Obtiene la lista de productos disponibles.")
def get_products():
    """Obtiene la lista de productos disponibles."""
    products = [
        {"name": "Teléfono inteligente" , "price": 1800},
        {"name": "Computadora portátil" , "price": 2500},
        {"name": "Auriculares" , "price": 300},
        {"name": "Reloj inteligente" , "price": 800},
    ]
    return "".join([f"{product['name']} - {product['price']} " for product in products])

In [ ]:
get_products.invoke({})

In [20]:
from langchain_core.tools import tool
import requests

@tool("get_products", description="Obtiene la lista de productos disponibles.")
def get_products():
    """Obtiene la lista de productos disponibles."""
    response = requests.get("https://api.escuelajs.co/api/v1/products")
    products = response.json()
    return "".join([f"{product['category']['name']} - {product['price']} " for product in products])

In [21]:
get_products.invoke({})

'nuevo - 28 nuevo - 48 nuevo - 10 nuevo - 43 nuevo - 97 nuevo - 39 Updated category name - 53 Updated category name - 24 Updated category name - 66 Updated category name - 25 Updated category name - 49 Updated category name - 71 Shoes - 39 Shoes - 39 Shoes - 33 Shoes - 27 Shoes - 84 Shoes - 68 Shoes - 36 Shoes - 53 Miscellaneous - 22 Miscellaneous - 37 Miscellaneous - 73 Miscellaneous - 48 Miscellaneous - 61 Miscellaneous - 38 '

In [16]:
from langchain_core.tools import tool
import requests
import dotenv
import os

load_dotenv()

@tool("get_horario", description="Obtiene la lista de horario disponible.")
def get_horario(id:str):
    """Obtiene la lista de horario disponible."""
    response = requests.get(f"{os.getenv('API_HORARIO_ACTUAL')}?cn={id}")
    horario = response.json()
    horario_unico = [dict(t) for t in {tuple(d.items()) for d in horario}]
    return horario_unico

In [ ]:
get_horario.invoke({"id": "000837050"})

In [21]:
system_prompt = """
Eres un asistente academico que ayuda a los estudiantes a obtener la informacion del horario de sus cursos.

Tus tools son:
- get_horario: Obtiene la lista de horario disponible.
"""

messages = [
    ("system", system_prompt),
    ("user", "¿Cual es mi horario, mi id es 000837050?")
]

llm_with_tools = llm.bind_tools([get_horario])
response = llm_with_tools.invoke(messages)
response.tool_calls

[{'name': 'get_horario',
  'args': {'id': '000837050'},
  'id': '57cf8960-670e-429e-bb46-62309dc5c346',
  'type': 'tool_call'}]